# bdap_entrate_stato - notebook v0

Validazione della pipeline per fasi: **raw → clean → mart**.

- scopo: verificare che ogni layer sia corretto e coerente con il precedente
- le SQL sono la fonte di verità: i check numerici devono essere letti alla luce di quello che dichiarano
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit

In [11]:
import re
import yaml
import pandas as pd
from pathlib import Path

# --- Unici parametri da impostare manualmente ---
METRICA       = "totale_cp"                 # colonna numerica principale nel mart
METRICA_CLEAN = "previsioni_definitive_cp"  # colonna corrispondente nel clean

# --- Anchor: usa il path del notebook se disponibile (VSCode), altrimenti cwd ---
try:
    _start = Path(__vsc_ipynb_file__).resolve().parent  # VSCode Jupyter
except NameError:
    _start = Path.cwd().resolve()

# Walk up finché non troviamo dataset.yml
candidate_dir = None
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        candidate_dir = probe
        break

if candidate_dir is None:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

cfg = yaml.safe_load((candidate_dir / "dataset.yml").read_text())

SLUG       = cfg["dataset"]["name"]
ANNO_RUN   = cfg["dataset"]["years"][-1]
MART_TABLE = cfg["mart"]["tables"][0]["name"]
ENCODING   = cfg.get("clean", {}).get("read", {}).get("encoding", "utf-8")
DELIM      = cfg.get("clean", {}).get("read", {}).get("delim", ",")

DI_ROOT   = (candidate_dir / cfg["root"]).resolve()
RAW_DIR   = DI_ROOT / "data" / "raw"   / SLUG / str(ANNO_RUN)
CLEAN_DIR = DI_ROOT / "data" / "clean" / SLUG / str(ANNO_RUN)
MART_DIR  = DI_ROOT / "data" / "mart"  / SLUG / str(ANNO_RUN)
SQL_DIR   = candidate_dir / "sql"

print(f"slug      : {SLUG}")
print(f"anno_run  : {ANNO_RUN}")
print(f"mart table: {MART_TABLE}")
print(f"encoding  : {ENCODING}  delim: {repr(DELIM)}")

slug      : bdap_entrate_stato
anno_run  : 2024
mart table: mart_entrate_titolo_natura_anno
encoding  : cp1252  delim: ';'


## SQL di riferimento

Le SQL sono la fonte di verità per capire cosa deve contenere ogni layer.
Leggile prima di interpretare i check numerici.

In [12]:
for sql_file in sorted(SQL_DIR.glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()

  clean.sql
select
  try_cast(column00 as integer) as esercizio_finanziario,
  nullif(trim(column01), '') as codice_titolo,
  nullif(trim(column02), '') as titolo,
  nullif(trim(column03), '') as codice_natura,
  nullif(trim(column04), '') as natura,
  nullif(trim(column05), '') as codice_tipologia,
  nullif(trim(column06), '') as tipologia,
  nullif(trim(column07), '') as codice_provento,
  nullif(trim(column08), '') as provento,
  try_cast(column09 as double) as previsioni_definitive_cp,
  try_cast(column10 as double) as previsioni_definitive_cs
from raw_input
where try_cast(column00 as integer) between 2008 and 2024
  and nullif(trim(column01), '') is not null
  and nullif(trim(column03), '') is not null
  and nullif(trim(column05), '') is not null


  mart.sql
with base as (
  select
    esercizio_finanziario as anno,
    codice_titolo,
    titolo,
    codice_natura,
    natura,
    sum(previsioni_definitive_cp) as totale_cp,
    sum(previsioni_definitive_cs) as totale_cs
  from cl

## 1. Raw

Verifica che il file raw esista, sia leggibile con i parametri dichiarati in `dataset.yml` e abbia il numero di righe atteso.

In [13]:
raw_files = sorted(RAW_DIR.glob("*.csv")) + sorted(RAW_DIR.glob("*.xlsx")) + sorted(RAW_DIR.glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

if raw_path.suffix == ".parquet":
    raw_full = pd.read_parquet(raw_path)
elif raw_path.suffix in (".csv", ".tsv"):
    raw_full = pd.read_csv(raw_path, encoding=ENCODING, sep=DELIM)
elif raw_path.suffix == ".xlsx":
    raw_full = pd.read_excel(raw_path)

N_RAW = len(raw_full)
print(f"Righe raw   : {N_RAW}")
print(f"Colonne raw : {len(raw_full.columns)} → {list(raw_full.columns)}")
raw_full.head(3)

File: bdap_entrate_stato_serie_storica.csv  (261 KB)
Righe raw   : 1320
Colonne raw : 12 → ['Esercizio Finanziario', 'Codice Titolo', 'Titolo', 'Codice Natura', 'Natura', 'Codice Tipologia', 'Tipologia', 'Codice Attività/Provento', 'Attività/Provento', 'Previsioni Definitive CP A1', 'Previsioni Definitive CS A1', 'Unnamed: 11']


,Esercizio Finanziario,Codice Titolo,Titolo,Codice Natura,Natura,Codice Tipologia,Tipologia,Codice Attività/Provento,Attività/Provento,Previsioni Definitive CP A1,Previsioni Definitive CS A1,Unnamed: 11
0,2008,1,TITOLO I - ENTRATE TRIBUTARIE,1,Entrate ricorrenti,1,Imposta sui redditi,1,Entrate derivanti dall'attivita' ordinaria di ...,1.653603e+11,1.653603e+11,NaN
1,2008,1,TITOLO I - ENTRATE TRIBUTARIE,1,Entrate ricorrenti,1,Imposta sui redditi,2,Entrate derivanti dall'attivita' di accertamen...,9.675500e+09,1.986500e+09,NaN
2,2008,1,TITOLO I - ENTRATE TRIBUTARIE,1,Entrate ricorrenti,10,Lotto,1,Entrate derivanti dall'attivita' ordinaria di ...,5.911000e+09,5.911000e+09,NaN


## 2. Clean

Verifica schema e null. I null su colonne `try_cast` sono attesi se il raw contiene valori non parsabili.
Il confronto righe raw vs clean mostra quante righe sono state filtrate dal `WHERE` della clean.sql.

In [14]:
clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
if not clean_files:
    raise FileNotFoundError(f"Clean non trovato in {CLEAN_DIR}. Eseguire: toolkit run clean")

clean = pd.read_parquet(clean_files[0])
N_CLEAN = len(clean)

print(f"Righe clean : {N_CLEAN}")
print(f"Colonne     : {list(clean.columns)}")
clean.head(3)

Righe clean : 1320
Colonne     : ['esercizio_finanziario', 'codice_titolo', 'titolo', 'codice_natura', 'natura', 'codice_tipologia', 'tipologia', 'codice_provento', 'provento', 'previsioni_definitive_cp', 'previsioni_definitive_cs']


,esercizio_finanziario,codice_titolo,titolo,codice_natura,natura,codice_tipologia,tipologia,codice_provento,provento,previsioni_definitive_cp,previsioni_definitive_cs
0,2008,1,TITOLO I - ENTRATE TRIBUTARIE,1,Entrate ricorrenti,1,Imposta sui redditi,1,Entrate derivanti dall'attivita' ordinaria di ...,1.653603e+11,1.653603e+11
1,2008,1,TITOLO I - ENTRATE TRIBUTARIE,1,Entrate ricorrenti,1,Imposta sui redditi,2,Entrate derivanti dall'attivita' di accertamen...,9.675500e+09,1.986500e+09
2,2008,1,TITOLO I - ENTRATE TRIBUTARIE,1,Entrate ricorrenti,10,Lotto,1,Entrate derivanti dall'attivita' ordinaria di ...,5.911000e+09,5.911000e+09


In [15]:
dropped = N_RAW - N_CLEAN
dropped_pct = dropped / N_RAW * 100 if N_RAW > 0 else 0

print(f"Righe raw         : {N_RAW}")
print(f"Righe clean       : {N_CLEAN}")
print(f"Righe filtrate    : {dropped} ({dropped_pct:.1f}%)")
print()
if dropped > 0:
    print("→ Verificare la WHERE in clean.sql per capire quali righe vengono escluse.")
else:
    print("→ Nessuna riga filtrata: clean è fedele al raw.")

print("\nNull per colonna clean:")
null_counts = clean.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "  nessuno")

Righe raw         : 1320
Righe clean       : 1320
Righe filtrate    : 0 (0.0%)

→ Nessuna riga filtrata: clean è fedele al raw.

Null per colonna clean:
previsioni_definitive_cp    139
previsioni_definitive_cs     90


## 3. Mart

Verifica unicità sulle chiavi del GROUP BY, anni presenti, null e consistenza dei totali con il clean.

In [16]:
mart_path = MART_DIR / f"{MART_TABLE}.parquet"
if not mart_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {mart_path}. Eseguire: toolkit run mart")

mart = pd.read_parquet(mart_path)
print(f"Righe mart  : {len(mart)}")
print(f"Colonne     : {list(mart.columns)}")
mart.head(3)

Righe mart  : 104
Colonne     : ['anno', 'codice_titolo', 'titolo', 'titolo_breve', 'codice_natura', 'natura', 'totale_cp', 'totale_cs', 'quota_cp', 'quota_cs']


,anno,codice_titolo,titolo,titolo_breve,codice_natura,natura,totale_cp,totale_cs,quota_cp,quota_cs
0,2008,1,TITOLO I - ENTRATE TRIBUTARIE,ENTRATE TRIBUTARIE,1,Entrate ricorrenti,4.514661e+11,4.321901e+11,0.616459,0.573338
1,2008,1,TITOLO I - ENTRATE TRIBUTARIE,ENTRATE TRIBUTARIE,2,Entrate non ricorrenti,2.115000e+09,2.115000e+09,0.002888,0.002806
2,2008,2,TITOLO II - ENTRATE EXTRA-TRIBUTARIE,ENTRATE EXTRA-TRIBUTARIE,1,Entrate ricorrenti,3.168345e+10,2.471669e+10,0.043262,0.032789


In [17]:
mart_sql_path = SQL_DIR / "mart.sql"
groupby_keys = []
if mart_sql_path.exists():
    sql_text = mart_sql_path.read_text()
    match = re.search(r'group\s+by\s+([\d\s,]+)', sql_text, re.IGNORECASE)
    if match:
        positions = [int(p.strip()) for p in match.group(1).split(',')]
        groupby_keys = [mart.columns[i - 1] for i in positions if i <= len(mart.columns)]
    else:
        match = re.search(r'group\s+by\s+((?:[\w.]+(?:\s*,\s*)?)+)', sql_text, re.IGNORECASE)
        if match:
            groupby_keys = [k.strip().split('.')[-1] for k in match.group(1).split(',')]
            groupby_keys = [k for k in groupby_keys if k in mart.columns]

if groupby_keys:
    dups = mart.duplicated(subset=groupby_keys).sum()
    print(f"Chiavi GROUP BY : {groupby_keys}")
    print(f"Duplicati       : {dups}")
    assert dups == 0, f"FAIL: {dups} righe duplicate sulle chiavi del mart — verificare mart.sql"
    print("OK: nessun duplicato sulle chiavi.")
else:
    print("GROUP BY non estratto automaticamente — verificare manualmente unicità.")

Chiavi GROUP BY : ['anno', 'codice_titolo', 'titolo', 'titolo_breve', 'codice_natura']
Duplicati       : 0
OK: nessun duplicato sulle chiavi.


In [18]:
if "anno" in mart.columns:
    anni_mart = sorted(mart["anno"].unique())
    print(f"Anni nel mart ({len(anni_mart)}): {anni_mart}")

null_mart = mart.isnull().sum()
print("\nNull per colonna mart:")
print(null_mart[null_mart > 0].to_string() if null_mart.any() else "  nessuno")

Anni nel mart (17): [np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

Null per colonna mart:
totale_cp    2
totale_cs    2
quota_cp     2
quota_cs     2


In [19]:
if METRICA in mart.columns and METRICA_CLEAN in clean.columns:
    tot_mart  = mart[METRICA].sum()
    tot_clean = clean[METRICA_CLEAN].sum()
    delta_pct = abs(tot_mart - tot_clean) / tot_clean * 100 if tot_clean != 0 else float("nan")
    print(f"Totale mart  ({METRICA})      : {tot_mart:,.2f}")
    print(f"Totale clean ({METRICA_CLEAN}): {tot_clean:,.2f}")
    print(f"Delta %: {delta_pct:.4f}%")
    assert delta_pct < 0.01, f"FAIL: delta {delta_pct:.2f}% — verificare la pipeline"
    print("OK: i totali coincidono.")
else:
    print(f"Colonne non trovate — aggiornare METRICA ('{METRICA}') e METRICA_CLEAN ('{METRICA_CLEAN}').")

Totale mart  (totale_cp)      : 15,690,764,002,724.80
Totale clean (previsioni_definitive_cp): 15,690,764,002,724.80
Delta %: 0.0000%
OK: i totali coincidono.


## Note v0

- Slug: `bdap_entrate_stato`
- Anno run: `2024`
- Tabella mart: `mart_entrate_titolo_natura_anno`
- Metrica guida mart: `totale_cp`
- Metrica clean: `previsioni_definitive_cp`
- Perimetro: Stato centrale, serie storica 2008-2024, livello nazionale